In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r'C:\Users\tasne\Desktop\Data Analyst\E-Commerce-Sales-Analysis\data\cleaned_data.csv')
df

In [ ]:
df['estimated_delivery_date'] = pd.to_datetime(df['estimated_delivery_date'])
df

In [ ]:
# calculate RFM
df['purchase_date'] = pd.to_datetime(df['purchase_date'])

snapshot_date = df['purchase_date'].max() + dt.timedelta(days=1)
rfm = df.groupby('customer_unique_id').agg({
    'purchase_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'order_id': 'nunique',                                    # Frequency
    'total_order_value': 'sum'                                # Monetary
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm

In [ ]:
df['purchase_date'] = pd.to_datetime(df['purchase_date'])
df

In [ ]:
# calculate RFM
snapshot_date = df['purchase_date'].max() + dt.timedelta(days=1)
rfm = df.groupby('customer_unique_id').agg({
    'purchase_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'order_id': 'nunique',                                    # Frequency
    'total_order_value': 'sum'                                # Monetary
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm

In [ ]:
pd.set_option('display.max_rows', None)
rfm

In [ ]:
# scoring
# Recency: less days high score
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
# Monetary: more money high score
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])
# Frequency: more orders high score
def f_scoring(x):
    if x == 1:
        return 1
    elif x == 2:
        return 3 
    else:
        return 5 
rfm['F_score'] = rfm['Frequency'].apply(f_scoring)
# final score
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str)
rfm

In [ ]:
# mapping
seg_map = {
    r'[1-2]1': 'Hibernating',         # very old one order
    r'[1-2]3': 'At Risk',            # very old two orders
    r'[1-2]5': 'Can\'t Loose Them',      # very old more orders
    r'31': 'About to Sleep',         # old one order
    r'33': 'Need Attention',             # old two orders
    r'[3-4]5': 'Loyal Customers',    # new more orders
    r'41': 'Promising',                  # new one order
    r'51': 'New Customers',              # very new one order
    r'[4-5]3': 'Potential Loyalists', # very ner two orders
    r'55': 'Champions'               # very new more orders 
}

rfm['Segment'] = rfm['RFM_Score'].replace(seg_map, regex=True)

rfm[['Recency', 'Frequency', 'Monetary', 'Segment']]

In [ ]:
# analysing
segment_summary = rfm.groupby('Segment').agg({
    'Monetary': ['mean', 'count'],
    'Recency': 'mean'
}).round(1)

segment_summary.columns = ['Avg_Revenue', 'Customer_Count', 'Avg_Recency']
segment_summary = segment_summary.sort_values(by='Customer_Count', ascending=False)

print(segment_summary)

In [ ]:
# view
plot_data = segment_summary.reset_index().sort_values(by='Customer_Count', ascending=False)
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
ax = sns.barplot(data=plot_data, x='Customer_Count', y='Segment', hue='Segment', palette='viridis', legend=False)
for i in ax.containers:
    ax.bar_label(i, padding=3)

plt.title('Distribution of Customer Segments', fontsize=15)
plt.xlabel('Number of Customers')
plt.ylabel('Segment')
plt.show()

In [ ]:
# exporting
final_data = df.merge(rfm[['Segment', 'RFM_Score']], on='customer_unique_id', how='left')
final_data.to_csv('olist_final_analysis.csv', index=False)